In [1]:
pip install spotifyscraper


  Obtaining dependency information for spotifyscraper from https://files.pythonhosted.org/packages/c4/95/22a70cadd41f73b95988a5818127ee80c9b83129418e960d089a712304a0/spotifyscraper-3.9.2-py3-none-any.whl.metadata
  Obtaining dependency information for httpx<1.0,>=0.27 from https://files.pythonhosted.org/packages/2a/39/e50c7c3a983047577ee07d2a9e53faf5a69493943ec3f6a384bdc792deb2/httpx-0.28.1-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/129.5 kB ? eta -:--:--
   ------------------ --------------------- 61.4/129.5 kB 1.7 MB/s eta 0:00:01
   ---------------------------------------- 129.5/129.5 kB 1.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/73.5 kB ? eta -:--:--
   ---------------------------------------- 73.5/73.5 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: httpx
    Found existing installation: httpx 0.26.0
    Uninstalling httpx-0.26.0:
      Successfully uninstalled httpx-0.26.0


In [5]:


import pandas as pd
import requests
from spotify_scraper import SpotifyClient


EDITORIAL_OWNER_HINTS = {"spotify"}

def is_user_generated(owner_name, owner_uri):
    owner_id = owner_uri.rsplit(":", 1)[-1].lower() if owner_uri else ""
    return not any(
        hint in owner_id or hint in owner_name.lower()
        for hint in EDITORIAL_OWNER_HINTS
    )

def find_user_playlists(keyword, count, client: SpotifyClient):
    results = client.search(keyword, types=("playlist",), limit=min(count * 2, 50))
    user_playlists = [
        pl for pl in results.playlists
        if pl.owner and is_user_generated(pl.owner.name, pl.owner.uri)
    ]
    return user_playlists[:count]

def collect_track_ids(playlists, client: SpotifyClient):
    per_playlist = {}
    for pl in playlists:
            full = client.get_playlist(pl.id, max_tracks=None)
            per_playlist[pl.id] = [
                entry.track.id for entry in full.tracks if entry.track and entry.track.id]
    return per_playlist

def build_dataframe_playlist(keyword, playlists, per_playlist):
    playlist_names = {pl.id: pl.name for pl in playlists}
    rows = [
        {
            "keyword": keyword,
            "playlist_name": playlist_names.get(playlist_id, ""),
            "playlist_id": playlist_id,
            "track_id": track_id,
        }
        for playlist_id, track_ids in per_playlist.items()
        for track_id in track_ids
    ]
    return pd.DataFrame(rows, columns=["keyword", "playlist_name", "playlist_id", "track_id"])



In [13]:
keyword="driving"
n_playlists=1
with SpotifyClient() as client:
        playlists = find_user_playlists(keyword, n_playlists, client)
        print(f"Using {len(playlists)} user playlists for {keyword!r}")
        per_playlist = collect_track_ids(playlists, client)

playlist_df = build_dataframe_playlist(keyword, playlists, per_playlist)
print(playlist_df.head())
print(f"{len(playlist_df)} rows, {playlist_df['track_id'].nunique()} unique track ids, "
          f"{playlist_df['playlist_id'].nunique()} playlists")


Using 1 user playlists for 'driving'
   keyword   playlist_name             playlist_id                track_id
0  driving  girl driving 🎀  1AiyR6siWYLcMOgek5DVr4  2ENexcMEMsYk0rVJigVD3i
1  driving  girl driving 🎀  1AiyR6siWYLcMOgek5DVr4  7CyPwkp0oE8Ro9Dd5CUDjW
2  driving  girl driving 🎀  1AiyR6siWYLcMOgek5DVr4  7ySUcLPVX7KudhnmNcgY2D
3  driving  girl driving 🎀  1AiyR6siWYLcMOgek5DVr4  3Vi5XqYrmQgOYBajMWSvCi
4  driving  girl driving 🎀  1AiyR6siWYLcMOgek5DVr4  1a73gcEg6h6Re6hHXoVltJ
108 rows, 108 unique track ids, 1 playlists


In [14]:


# List of Spotify track IDs
spotify_ids = playlist_df['track_id'].unique().tolist()

results = []

for spotify_id in spotify_ids:
    try:
        lookup_resp = requests.get(
            "https://api.reccobeats.com/v1/track",
            params={"ids": spotify_id}
        )
        lookup_resp.raise_for_status()
        lookup_data = lookup_resp.json()

        if not lookup_data.get("content"):
            print(f"No track found for {spotify_id}")
            continue

        # Track metadata
        track = lookup_data["content"][0]

        reccobeats_id = track["id"]
        track_name = track["trackTitle"]

        # Get artist name
        artist_name = ", ".join(
            artist["name"] for artist in track["artists"]
        )

        # Get audio features
        features_resp = requests.get(
            f"https://api.reccobeats.com/v1/track/{reccobeats_id}/audio-features"
        )
        features_resp.raise_for_status()
        features = features_resp.json()

        # Add metadata to audio features
        features["spotify_id"] = spotify_id
        features["track_name"] = track_name
        features["artist"] = artist_name
        features["reccobeats_id"] = reccobeats_id

        results.append(features)

        print(f"Processed: {artist_name} - {track_name}")

    except requests.exceptions.RequestException as e:
        print(f"Error processing {spotify_id}: {e}")

# Create DataFrame
track_df = pd.DataFrame(results)
id_columns = [
    "spotify_id",
    "track_name",
    "artist",
    "reccobeats_id"
]

other_columns = [
    col for col in track_df.columns
    if col not in id_columns
]

track_df = track_df[id_columns + other_columns]

print(track_df)

Error processing 2ENexcMEMsYk0rVJigVD3i: 520 Server Error: <none> for url: https://api.reccobeats.com/v1/track?ids=2ENexcMEMsYk0rVJigVD3i
Processed: The Weeknd, JENNIE, Lily-Rose Depp - One Of The Girls (with JENNIE, Lily Rose Depp)
Processed: Rihanna - S&M
Processed: Doja Cat - Need to Know
Error processing 1a73gcEg6h6Re6hHXoVltJ: 520 Server Error: <none> for url: https://api.reccobeats.com/v1/track/df5eef09-4f32-41ae-9267-250ac2f4082d/audio-features
Processed: Flo Milli - Never Lose Me
Processed: USHER - Hey Daddy (Daddy's Home)
No track found for 3DZQ6mzUkAdHqZWzqxBKIK
Processed: Icona Pop, Charli xcx, Charlie XCX - I Love It (feat. Charli XCX)
No track found for 3oiMJQAWVaxSubJ7b2VUtX
Processed: Drake, Sexyy Red, SZA - Rich Baby Daddy (feat. Sexyy Red & SZA)
No track found for 2tpfxAXiI52znho4WE3XFA
Processed: Bruno Mars - That's What I Like
Processed: Katy Perry - Teenage Dream
No track found for 06EL94D0TA27Ik0Ke5usbj
Processed: USHER, Pitbull, Almighty Associates, Usher featurin

In [17]:
playlist_df.head()

,keyword,playlist_name,playlist_id,track_id
0,driving,girl driving 🎀,1AiyR6siWYLcMOgek5DVr4,2ENexcMEMsYk0rVJigVD3i
1,driving,girl driving 🎀,1AiyR6siWYLcMOgek5DVr4,7CyPwkp0oE8Ro9Dd5CUDjW
2,driving,girl driving 🎀,1AiyR6siWYLcMOgek5DVr4,7ySUcLPVX7KudhnmNcgY2D
3,driving,girl driving 🎀,1AiyR6siWYLcMOgek5DVr4,3Vi5XqYrmQgOYBajMWSvCi
4,driving,girl driving 🎀,1AiyR6siWYLcMOgek5DVr4,1a73gcEg6h6Re6hHXoVltJ


In [16]:
track_df.head()

,spotify_id,track_name,artist,reccobeats_id,id,href,isrc,acousticness,danceability,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,valence
0,7CyPwkp0oE8Ro9Dd5CUDjW,"One Of The Girls (with JENNIE, Lily Rose Depp)","The Weeknd, JENNIE, Lily-Rose Depp",1539ffae-748f-4ecc-b30b-874b3236c2db,1539ffae-748f-4ecc-b30b-874b3236c2db,https://open.spotify.com/track/7CyPwkp0oE8Ro9D...,USUG12303881,0.039300,0.264,0.640,0.000002,8,0.1510,-6.221,1,0.0442,84.828,0.1900
1,7ySUcLPVX7KudhnmNcgY2D,S&M,Rihanna,3d4c8ee7-e569-41d1-857e-f831e3c03fdf,3d4c8ee7-e569-41d1-857e-f831e3c03fdf,https://open.spotify.com/track/7ySUcLPVX7Kudhn...,USUM71026591,0.010500,0.769,0.685,0.000124,1,0.0887,-5.019,1,0.0410,127.977,0.8360
2,3Vi5XqYrmQgOYBajMWSvCi,Need to Know,Doja Cat,47cf3f3c-336c-4c6c-9913-0c07a005394d,47cf3f3c-336c-4c6c-9913-0c07a005394d,https://open.spotify.com/track/3Vi5XqYrmQgOYBa...,USRC12101120,0.304000,0.664,0.609,0.000000,1,0.0926,-6.509,1,0.0707,130.041,0.1940
3,6uTPdRrEDeH8Fyg5L5qmeU,Never Lose Me,Flo Milli,0f230283-ab2b-478b-9067-a613a143663a,0f230283-ab2b-478b-9067-a613a143663a,https://open.spotify.com/track/6uTPdRrEDeH8Fyg...,USRC12303379,0.050400,0.781,0.567,0.000000,1,0.0981,-7.857,1,0.0969,121.964,0.0743
4,030OCtLMrljNhp8OWHBWW3,Hey Daddy (Daddy's Home),USHER,9007abe1-9e42-420b-80a4-7684d353f328,9007abe1-9e42-420b-80a4-7684d353f328,https://open.spotify.com/track/030OCtLMrljNhp8...,USLF21000011,0.000176,0.590,0.698,0.000000,11,0.1070,-4.262,1,0.0286,95.975,0.3520


In [20]:
joined_table=track_df.merge(playlist_df, left_on="spotify_id", right_on="track_id", how="left")

In [21]:
joined_table.head()

,spotify_id,track_name,artist,reccobeats_id,id,href,isrc,acousticness,danceability,energy,...,liveness,loudness,mode,speechiness,tempo,valence,keyword,playlist_name,playlist_id,track_id
0,7CyPwkp0oE8Ro9Dd5CUDjW,"One Of The Girls (with JENNIE, Lily Rose Depp)","The Weeknd, JENNIE, Lily-Rose Depp",1539ffae-748f-4ecc-b30b-874b3236c2db,1539ffae-748f-4ecc-b30b-874b3236c2db,https://open.spotify.com/track/7CyPwkp0oE8Ro9D...,USUG12303881,0.039300,0.264,0.640,...,0.1510,-6.221,1,0.0442,84.828,0.1900,driving,girl driving 🎀,1AiyR6siWYLcMOgek5DVr4,7CyPwkp0oE8Ro9Dd5CUDjW
1,7ySUcLPVX7KudhnmNcgY2D,S&M,Rihanna,3d4c8ee7-e569-41d1-857e-f831e3c03fdf,3d4c8ee7-e569-41d1-857e-f831e3c03fdf,https://open.spotify.com/track/7ySUcLPVX7Kudhn...,USUM71026591,0.010500,0.769,0.685,...,0.0887,-5.019,1,0.0410,127.977,0.8360,driving,girl driving 🎀,1AiyR6siWYLcMOgek5DVr4,7ySUcLPVX7KudhnmNcgY2D
2,3Vi5XqYrmQgOYBajMWSvCi,Need to Know,Doja Cat,47cf3f3c-336c-4c6c-9913-0c07a005394d,47cf3f3c-336c-4c6c-9913-0c07a005394d,https://open.spotify.com/track/3Vi5XqYrmQgOYBa...,USRC12101120,0.304000,0.664,0.609,...,0.0926,-6.509,1,0.0707,130.041,0.1940,driving,girl driving 🎀,1AiyR6siWYLcMOgek5DVr4,3Vi5XqYrmQgOYBajMWSvCi
3,6uTPdRrEDeH8Fyg5L5qmeU,Never Lose Me,Flo Milli,0f230283-ab2b-478b-9067-a613a143663a,0f230283-ab2b-478b-9067-a613a143663a,https://open.spotify.com/track/6uTPdRrEDeH8Fyg...,USRC12303379,0.050400,0.781,0.567,...,0.0981,-7.857,1,0.0969,121.964,0.0743,driving,girl driving 🎀,1AiyR6siWYLcMOgek5DVr4,6uTPdRrEDeH8Fyg5L5qmeU
4,030OCtLMrljNhp8OWHBWW3,Hey Daddy (Daddy's Home),USHER,9007abe1-9e42-420b-80a4-7684d353f328,9007abe1-9e42-420b-80a4-7684d353f328,https://open.spotify.com/track/030OCtLMrljNhp8...,USLF21000011,0.000176,0.590,0.698,...,0.1070,-4.262,1,0.0286,95.975,0.3520,driving,girl driving 🎀,1AiyR6siWYLcMOgek5DVr4,030OCtLMrljNhp8OWHBWW3
